## Import of Libraries

In [1]:
# Import visualization libraries
import seaborn as sns
import matplotlib.pyplot as plt

# Import data handling libraries
import pandas as pd
import numpy as np
import pickle

# Import machine learning libraries
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import euclidean_distances
from sklearn.metrics import silhouette_score
from sklearn.metrics import silhouette_samples
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import Normalizer
from sklearn.neighbors import NearestNeighbors

## Configuration of Display Options

In [2]:
# Set display format for floats to 2 decimal places
pd.set_option("display.float_format", "{:,.2f}".format)

## Loading of Dataset

In [3]:
# Import preprocessor
with open("../data/processed/preprocessor.pkl", "rb") as f:
    preprocessor = pickle.load(f)

# Import preprocessed feature matrices
with open("../data/processed/X_train_preprocessed.pkl", "rb") as f:
    X_train_preprocessed = pickle.load(f)
with open("../data/processed/X_test_preprocessed.pkl", "rb") as f:
    X_test_preprocessed = pickle.load(f)

# Import feature names
with open("../data/processed/feature_names.pkl", "rb") as f:
    feature_names = pickle.load(f)

In [4]:
# Plot names of preporcessed features
display("Preprocessed Features")
for name in feature_names: 
    print(name)

'Preprocessed Features'

minmax__popularity
standard__duration_min
standard__loudness
standard__tempo
standard__power_score
standard__chill_score
standard__groove
cat__track_genre_ambient
cat__track_genre_breakbeat
cat__track_genre_chicago-house
cat__track_genre_club
cat__track_genre_dance
cat__track_genre_deep-house
cat__track_genre_drum-and-bass
cat__track_genre_dubstep
cat__track_genre_edm
cat__track_genre_electro
cat__track_genre_electronic
cat__track_genre_hardstyle
cat__track_genre_house
cat__track_genre_idm
cat__track_genre_minimal-techno
cat__track_genre_progressive-house
cat__track_genre_synth-pop
cat__track_genre_techno
cat__track_genre_trance
cat__camelot_id_10A
cat__camelot_id_10B
cat__camelot_id_11A
cat__camelot_id_11B
cat__camelot_id_12A
cat__camelot_id_12B
cat__camelot_id_1A
cat__camelot_id_1B
cat__camelot_id_2A
cat__camelot_id_2B
cat__camelot_id_3A
cat__camelot_id_3B
cat__camelot_id_4A
cat__camelot_id_4B
cat__camelot_id_5A
cat__camelot_id_5B
cat__camelot_id_6A
cat__camelot_id_6B
cat__camelot_id

## Baseline model: k-Means

### Instantiate Model

In [6]:
# Create clusters of five tracks each
model_km = KMeans(n_clusters=len(X_train_preprocessed)/5,
                  random_state=42)

### Organize data in feature matrix

In [7]:
# Select columns for feature selection
selected_exact = [
    "raw__danceability",
    "raw__energy",
    "remainder__speechiness",
    "raw__instrumentalness",
    "raw__valence",
    "standard__tempo",
]

camelot_prefix = "cat__camelot__id_"

# Create masks for filtering data
mask_exact = np.isin(feature_names, selected_exact)
mask_camelot = np.char.startswith(feature_names.astype(str), camelot_prefix)

mask = mask_exact | mask_camelot

# Perform feature selection
X_train_sel = X_train_preprocessed[:, mask]
X_test_sel = X_test_preprocessed[:, mask]

### Adjust model to the data

In [ ]:
# Fit model
model_km.fit(X_train)

# Evaluate quality of selected number of clusters using WCSS (within cluster sum of squares)
model_km.score(X_train)

# Predict classifications and add labels
labels_train = model_km.predict(X_train)
# labels_train = model_km.labels_
# df['Model_KM_Labels'] = labels_train
labels_test = model_km.predict(X_test)

# Determine the Euclidean distances
distances_train = model_km.transform(X_train)
# distances = euclidean_distances(X_train, model_km.cluster_centers_)
distances_test = model_km.transform(X_test)

In [ ]:
def fit_kmeans(X, k):
    if sparse.issparse(X):
        return MiniBatchKMeans(n_clusters=k, 
                               random_state=42, 
                               n_init=10, 
                               batch_size=2048).fit(X)
    else:
        return KMeans(n_clusters=k, random_state=42, n_init=10).fit(X)

k_example = 8
model_km = fit_kmeans(X_train, k_example)

# Labels auf Train/Test
labels_train = model_km.labels_
labels_test  = model_km.predict(X_test)

# Clustergrößen
import numpy as np
unique, counts = np.unique(labels_train, return_counts=True)
print("Clustergrößen (Train):", dict(zip(unique, counts)))

### Evaluation of model quality

In [ ]:
# Determining k: Elbow-Method
# Find the best number of clusters
cluster_scores = []
for n_cluster in range(1, len(X_train)):)
    cluster_scores.append(model_km.score(X_train))

fig, ax = plt.subplots()
ax.plot(range(1, len(X_train)), cluster_scores, marker='o', markersize=7)
ax.set_title("Finding the best value for k")
ax.set_xlabel("k")
ax.set_ylabel("Within-cluster sum of squares")

In [ ]:
k_values = [2, 4, 6, 8, 10, 12]
inertias = []

for k in k_values:
    m = fit_kmeans(X_train, k)
    inertias.append(m.inertia_)  # Sum of squared distances

plt.figure()
plt.plot(k_values, inertias, marker="o")
plt.xlabel("Anzahl Cluster k")
plt.ylabel("Inertia (Sum of Squared Distances)")
plt.title("Elbow-Methode")
plt.show()

In [ ]:
# Interpretation of clusters
for n_init in [1, 10, 50]:
    cluster_scores  = []
    for i in range(10):
        model_km = KMeans(n_clusters=7, n_init=n_init)
        model_km.fit(X_train)
        cluster_scores.append(model_km.score(X_train))
    print(np.mean(cluster_scores), np.std(cluster_scores))